# TV Script Generation

In this lab, you'll generate your own [Seinfeld](https://en.wikipedia.org/wiki/Seinfeld) TV scripts using RNNs.  You'll be using part of the [Seinfeld dataset](https://www.kaggle.com/thec03u5/seinfeld-chronicles#scripts.csv) of scripts from 9 seasons.  The neural network you'll build will generate a new ,"fake" TV script, based on patterns it recognizes in this training data.

## Data

The data is already provided for you in the lab files and you're encouraged to open that file and look at the text. 
* As a first step, we'll load in this data and look at some samples. 
* Then, you'll be tasked with defining and training an RNN to generate a new script!

In [ ]:
# self-contained notebook utilities
import os
import pickle
import re
import torch
import numpy as np

SPECIAL_WORDS = {'PADDING': '<PAD>'}

def strip_optional_script_notice(raw_text):
    """
    We skip anything before the first line matching ``speaker:`` (letters/digits/underscore/hyphen).
    """
    raw_text = raw_text.lstrip("\ufeff")
    match = re.search(r"^[a-zA-Z][a-zA-Z0-9_-]*\s*:", raw_text, re.MULTILINE)
    if match:
        return raw_text[match.start() :]
    return raw_text


def load_data(path):
    """Load dataset text from file."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()


def preprocess_and_save_data(dataset_path, token_lookup, create_lookup_tables):
    """Preprocess and persist tokenized text + vocab artifacts."""
    text = strip_optional_script_notice(load_data(dataset_path))

    token_dict = token_lookup()
    for key, token in token_dict.items():
        text = text.replace(key, f" {token} ")

    text = text.lower().split()

    vocab_to_int, int_to_vocab = create_lookup_tables(text + list(SPECIAL_WORDS.values()))
    int_text = [vocab_to_int[word] for word in text]

    with open("preprocess.p", "wb") as f:
        pickle.dump(
            (int_text, vocab_to_int, int_to_vocab, token_dict),
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )


def load_preprocess():
    """Load preprocessed artifacts from disk."""
    with open("preprocess.p", "rb") as f:
        return pickle.load(f)


def _checkpoint_path(filename):
    """Resolve ``./save/foo`` -> ``./save/foo.pt`` (same folder as logical basename)."""
    base_dir = os.path.dirname(filename)
    stem = os.path.splitext(os.path.basename(filename))[0] + ".pt"
    return os.path.join(base_dir, stem) if base_dir else stem


def save_model(filename, model):
    out_path = _checkpoint_path(filename)
    parent = os.path.dirname(out_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    torch.save(model, out_path)


def load_model(filename):
    # Full-module pickle loading used by this lab workflow.
    out_path = _checkpoint_path(filename)
    return torch.load(out_path, map_location="cpu", weights_only=False)


data_dir = "data/Seinfeld_short.txt"  # TODO: point this at your dataset
text = load_data(data_dir)

## Explore the Data

In [ ]:
print('Dataset Stats')
print('Roughly the number of unique words: {}'.format(len({word: None for word in text.split()})))

lines = text.split('\n')
print('Number of lines: {}'.format(len(lines)))
word_count_line = [len(line.split()) for line in lines]
print('Average number of words in each line: {}'.format(np.average(word_count_line)))


view_line_range = (0, 10)
print('The lines {} to {}:'.format(*view_line_range))
print('\n'.join(text.split('\n')[view_line_range[0]:view_line_range[1]]))

Dataset Stats
Roughly the number of unique words: 28140
Number of lines: 54617
Average number of words in each line: 5.7282531080066645
The lines 0 to 10:
jerry: do you know what this is all about? do you know, why were here? to be out, this is out...and out is one of the single most enjoyable experiences of life. people...did you ever hear people talking about we should go out? this is what theyre talking about...this whole thing, were all out now, no one is home. not one person here is home, were all out! there are people trying to find us, they dont know where we are. (on an imaginary phone) did you ring?, i cant find him. where did he go? he didnt tell me where he was going. he must have gone out. you wanna go out you get ready, you pick out the clothes, right? you take the shower, you get all ready, get the cash, get your friends, the car, the spot, the reservation...then youre standing around, what do you do? you go we gotta be getting back. once youre out, you wanna get back! yo

---
## Pre-processing Functions
Implement the following text preprocessing functions:
- Lookup Table
- Tokenize Punctuation

### Lookup Table
Before we can look words up in an embedding table, we need integer ids for each token. In this function, create two dictionaries:
- Dictionary to map word -> id (`vocab_to_int`)
- Dictionary to map id -> word (`int_to_vocab`)

Return these dictionaries as a tuple: `(vocab_to_int, int_to_vocab)`

In [ ]:
def create_lookup_tables(text):
    """
    Create lookup tables for vocabulary.

    :param text: The text of tv scripts split into words (list of str).
    :return: A tuple of dicts (vocab_to_int, int_to_vocab).
    """
    # Stable sort by frequency descending, with alphabetical tie-break, so the
    # lookup tables are deterministic across runs.
    from collections import Counter

    counts = Counter(text)
    sorted_words = sorted(counts.keys(), key=lambda w: (-counts[w], w))
    vocab_to_int = {word: idx for idx, word in enumerate(sorted_words)}
    int_to_vocab = {idx: word for word, idx in vocab_to_int.items()}
    return vocab_to_int, int_to_vocab


### Tokenize Punctuation
We'll split the script using spaces. Without tokenization, punctuation can create separate variants of the same word (for example, `bye` vs `bye!`).

Implement `token_lookup` to return a dict that maps punctuation symbols (for example `!`) to tokens (for example `||Exclamation_Mark||`). This lets the model learn punctuation as its own token while keeping word forms more consistent.

Create entries for:
- Period ( **.** )
- Comma ( **,** )
- Quotation Mark ( **"** )
- Semicolon ( **;** )
- Exclamation mark ( **!** )
- Question mark ( **?** )
- Left Parentheses ( **(** )
- Right Parentheses ( **)** )
- Dash ( **-** )
- Return ( **\n** )

Use token values that cannot be confused with normal words, for example `||dash||`.

In [ ]:
def token_lookup():
    """
    Generate a dict mapping punctuation to tokens.

    :return: Dict where the key is the punctuation and the value is the token.
    """
    return {
        ".": "||period||",
        ",": "||comma||",
        '"': "||quotation_mark||",
        ";": "||semicolon||",
        "!": "||exclamation_mark||",
        "?": "||question_mark||",
        "(": "||left_parentheses||",
        ")": "||right_parentheses||",
        "-": "||dash||",
        "\n": "||return||",
    }


## Pre-process all the data and save it

Running the code cell below will pre-process all the data and save it to file using the utility functions defined earlier in this notebook.

In [ ]:
preprocess_and_save_data(data_dir, token_lookup, create_lookup_tables)

In [ ]:
int_text, vocab_to_int, int_to_vocab, token_dict = load_preprocess()
assert SPECIAL_WORDS['PADDING'] in vocab_to_int, "Vocabulary must contain the <PAD> token"

## Training a RNN/LSTM
In this section, you'll build the components necessary to build an RNN by implementing the RNN Module and forward and backpropagation functions.

### Device (CPU, CUDA, or MPS)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
torch.manual_seed(42)
np.random.seed(42)

## Sequence batching
Build a `DataLoader` of `(inputs, targets)` pairs from the one-dimensional token-id stream produced by preprocessing. We are training a **language model**, so each sample has shape `(T,)` and a full batch has shape `(B, T)`, where `T = sequence_length` and `B = batch_size`. Targets are the inputs shifted by one position.

### Stride-1 sliding window
Implement `batch_data(words, sequence_length, batch_size)` so that, for each starting index `i`:

- `inputs[i]  = words[i : i + T]`
- `targets[i] = words[i + 1 : i + 1 + T]`

Example for `words = [1, 2, 3, 4, 5, 6, 7]` and `T = 4`:

- `inputs = [1, 2, 3, 4]   targets = [2, 3, 4, 5]`
- `inputs = [2, 3, 4, 5]   targets = [3, 4, 5, 6]`
- `inputs = [3, 4, 5, 6]   targets = [4, 5, 6, 7]`

You can build at most `len(words) - T` pairs: any window starting after that index would need a target word that does not exist.

`shuffle=True` is fine in the DataLoader for training, since each sample already carries its own local sequential context.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def batch_data(words, sequence_length: int, batch_size: int) -> DataLoader:
    """Build a DataLoader of (inputs, targets) pairs for causal LM training.

    Each sample is a stride-1 sliding window over ``words``:

        inputs[i]  = words[i     : i + T]
        targets[i] = words[i + 1 : i + 1 + T]

    where T = sequence_length. The dataset contains exactly
    ``len(words) - T`` windows.
    """
    words_tensor = torch.as_tensor(words, dtype=torch.long)
    n_windows = words_tensor.size(0) - sequence_length
    if n_windows <= 0:
        raise ValueError(
            f"Not enough tokens ({words_tensor.size(0)}) for sequence_length="
            f"{sequence_length}."
        )

    # Materialise inputs/targets as (n_windows, T) tensors.
    inputs = torch.stack([
        words_tensor[i : i + sequence_length] for i in range(n_windows)
    ])
    targets = torch.stack([
        words_tensor[i + 1 : i + 1 + sequence_length] for i in range(n_windows)
    ])

    dataset = TensorDataset(inputs, targets)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
    )


In [ ]:
# Quick verification: build a small loader and inspect one batch.
# Expected:
#   - inputs.shape  == (B, T)
#   - targets.shape == (B, T)
#   - targets[i, :-1] == inputs[i, 1:]   (shift-by-one rule)
_demo_loader = batch_data(int_text, sequence_length=10, batch_size=4)
_inputs, _targets = next(iter(_demo_loader))
print('inputs  shape:', tuple(_inputs.shape), 'dtype:', _inputs.dtype)
print('targets shape:', tuple(_targets.shape), 'dtype:', _targets.dtype)
print('inputs [0]   :', _inputs[0].tolist())
print('targets[0]   :', _targets[0].tolist())
assert torch.equal(_targets[0, :-1], _inputs[0, 1:]), "targets must equal inputs shifted by 1"

---
## Build the RNN/LSTM model
Implement a word-level language model with PyTorch's [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). You may choose **GRU** or **LSTM**.

Implement:
- `__init__`
- `init_hidden`
- `forward`

### Training objective: predict the next token at every step
For an input window `inputs = (B, T)` and `targets = (B, T)` shifted by one position, the model returns logits of shape `(B, T, V)`. The loss is computed on **all** time steps (`B * T` predictions per batch), the standard language-modeling objective. Note that we will compute the loss in `optimization_step`, so just return the full logits here.

### Required forward shapes
- `nn_input`: `(B, T)` token IDs
- `embeds`:  `(B, T, embedding_dim)`
- `rnn_out`: `(B, T, hidden_dim)`
- `logits`:  `(B, T, vocab_size)`

Return `(logits, hidden)`. **Do not** slice the final time step; the training step will flatten with `logits.view(-1, V)` and `targets.view(-1)`.

In [ ]:
import torch.nn as nn
from torch import Tensor
from typing import Tuple, Union

# RNN/GRU hidden: Tensor. LSTM hidden: tuple(h_0, c_0).
RNNHidden = Union[Tensor, Tuple[Tensor, Tensor]]


class RNN(nn.Module):
    """Word-level next-token language model with an LSTM backbone."""

    def __init__(
        self,
        vocab_size: int,
        output_size: int,
        embedding_dim: int,
        hidden_dim: int,
        n_layers: int,
        dropout: float = 0.5,
    ) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.output_size = output_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, nn_input: Tensor, hidden: RNNHidden) -> Tuple[Tensor, RNNHidden]:
        """Compute next-token logits at every timestep.

        Shape flow: (B, T) -> (B, T, E) -> (B, T, H) -> (B, T, V)
        """
        embeds = self.embedding(nn_input)              # (B, T, E)
        out, hidden = self.lstm(embeds, hidden)        # (B, T, H)
        out = self.dropout(out)
        logits = self.fc(out)                          # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size: int) -> RNNHidden:
        """Initial recurrent state (zeros on the parameter device/dtype)."""
        device = next(self.parameters()).device
        dtype = next(self.parameters()).dtype
        h0 = torch.zeros(self.n_layers, batch_size, self.hidden_dim, dtype=dtype, device=device)
        c0 = torch.zeros(self.n_layers, batch_size, self.hidden_dim, dtype=dtype, device=device)
        return (h0, c0)


### Define the optimization step
Implement `optimization_step` to run one training step and return `(loss.item(), hidden)`.

```
loss, hidden = optimization_step(rnn, optimizer, criterion, inp, target, hidden)
```

Important details for this exercise:
- move `inp`, `target`, and `hidden` to `device`
- detach hidden state between batches to avoid backpropagating through the entire epoch graph
- apply gradient clipping before `optimizer.step()` for training stability

In [ ]:
def optimization_step(rnn, optimizer, criterion, inp, target, hidden):
    """Run one optimization step of the causal language model.

    Returns ``(loss_value, new_hidden)`` where ``loss_value`` is averaged over
    ``batch * seq_len`` next-token predictions.
    """
    rnn.train()
    inp = inp.to(next(rnn.parameters()).device)
    target = target.to(next(rnn.parameters()).device)

    # Detach hidden state from the previous batch's graph.
    if isinstance(hidden, tuple):
        hidden = tuple(h.detach() for h in hidden)
    else:
        hidden = hidden.detach()

    optimizer.zero_grad()
    logits, hidden = rnn(inp, hidden)                  # (B, T, V)

    # Flatten for CE: logits (B*T, V), target (B*T,)
    B, T, V = logits.shape
    loss = criterion(logits.reshape(B * T, V), target.reshape(B * T))

    loss.backward()
    nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=5.0)
    optimizer.step()

    return loss.item(), hidden


## Training Loop
The training loop is provided in `train_loop`. It iterates over batches for `n_epochs`.

### Hyperparameters

Set and train the neural network with the following parameters:
- Set `sequence_length` to the length of a sequence.
- Set `batch_size` to the batch size.
- Set `num_epochs` to the number of epochs to train for.
- Set `learning_rate` to the learning rate for an Adam optimizer.
- Set `vocab_size` to the number of unique tokens in our vocabulary.
- Set `output_size` to the desired size of the output.
- Set `embedding_dim` to the embedding dimension; smaller than the vocab_size.
- Set `hidden_dim` to the hidden dimension of your RNN.
- Set `n_layers` to the number of layers/cells in your RNN.
- Set `show_every_n_batches` to the number of batches at which the neural network should print progress.

If the network isn't getting the desired results, tweak these parameters and/or the layers in the `RNN` class.

In [ ]:
# Data params (tune as needed after your implementations work)
sequence_length = 10  # number of consecutive word ids per input sequence
batch_size = 128
train_loader = batch_data(int_text, sequence_length, batch_size)
# Training parameters
num_epochs = 1  # increase once training is stable
learning_rate = 0.001
# Model parameters (must match loaded preprocessing)
vocab_size = len(vocab_to_int)
output_size = vocab_size  # one score per vocabulary word (next-word prediction)
embedding_dim = 128
hidden_dim = 128
n_layers = 2
show_every_n_batches = 500

In [ ]:
def train_loop(rnn, batch_size, optimizer, criterion, n_epochs, show_every_n_batches=100):
    # ``train_loader`` is configured with ``drop_last=True`` in ``batch_data``,
    # so every batch is guaranteed to have exactly ``batch_size`` rows and we
    # can keep the hidden state initialized once per epoch.
    batch_losses = []
    rnn.train()

    print(f"Training for {n_epochs} epoch(s)...")
    for epoch_i in range(1, n_epochs + 1):

        # initialize hidden state
        hidden = rnn.init_hidden(batch_size)

        for batch_i, (inputs, labels) in enumerate(train_loader, 1):
            # forward, back prop
            loss, hidden = optimization_step(rnn, optimizer, criterion, inputs, labels, hidden)
            # record loss
            batch_losses.append(loss)

            # printing loss stats
            if batch_i % show_every_n_batches == 0:
                print('Epoch: {:>4}/{:<4}  Loss: {}\n'.format(
                    epoch_i, n_epochs, np.average(batch_losses)))
                batch_losses = []

    # returns a trained rnn
    return rnn

### Optional: Quick overfit sanity check
Before full training, you can sanity-check your implementation by overfitting on a small subset (for example first 1000 tokens). If loss does not go down quickly here, debug before long runs.

In [ ]:
# Quick overfit sanity check on the first 1000 tokens.
# After implementing batch_data / RNN / optimization_step, this short loop should
# drive the loss far below ln(vocab_size). If it doesn't, debug here before
# launching a full training run.
small_batch_size = 32
small_loader = batch_data(int_text[:1000], sequence_length, small_batch_size)

debug_rnn = RNN(vocab_size, output_size, embedding_dim, hidden_dim, n_layers, dropout=0.5).to(device)
debug_optim = torch.optim.Adam(debug_rnn.parameters(), lr=learning_rate)
debug_crit = nn.CrossEntropyLoss()

debug_rnn.train()
hidden = debug_rnn.init_hidden(small_batch_size)
N=5
for epoch in range(N):
    print(f"Epoch {epoch+1} of {N}")
    hidden = debug_rnn.init_hidden(small_batch_size)
    for step, (inp, tgt) in enumerate(small_loader, 1):
        loss, hidden = optimization_step(debug_rnn, debug_optim, debug_crit, inp, tgt, hidden)
        if step % 5 == 0:
            print(f"step {step:>4}  loss {loss:.4f}")

### Full Training
Train the network on preprocessed data. If optimization is unstable, revisit hyperparameters and apply gradient clipping in `optimization_step`.

This loss is cross-entropy (nats/token). As a reference, random guessing over vocabulary size `V` has loss about `ln(V)` — for the Seinfeld corpus used here `V` is roughly 28k, so a fully untrained model should sit near `ln(28000) ≈ 10.2`.

Try different `sequence_length` values to explore short vs. longer context dependencies.

In [ ]:
# create model and move to device
rnn = RNN(vocab_size, output_size, embedding_dim, hidden_dim, n_layers, dropout=0.5)
rnn = rnn.to(device)

# defining loss and optimization functions for training
optimizer = torch.optim.Adam(rnn.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# training the model
trained_rnn = train_loop(rnn, batch_size, optimizer, criterion, num_epochs, show_every_n_batches)

# saving the trained model
save_model('./save/trained_rnn', trained_rnn)
print('Model Trained and Saved')

### Question: How did you decide on your model hyperparameters? 
For example, did you try different sequence_lengths and find that one size made the model converge faster? What about your hidden_dim and n_layers; how did you decide on those?

---
# Checkpoint
If you pause here, re-run the load cell below to restore preprocessing artifacts and your saved model.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

_, vocab_to_int, int_to_vocab, token_dict = load_preprocess()
trained_rnn = load_model('./save/trained_rnn')
trained_rnn = trained_rnn.to(device)

## Generate TV Script
With the network trained and saved, you'll use it to generate a new, "fake" Seinfeld TV script in this section.

### Generate Text
To generate the text, the network needs to start with a single word and repeat its predictions until it reaches a set length. You'll be using the `generate` function to do this. It takes a word id to start with, `prime_id`, and generates a set length of text, `predict_len`. Also note that it uses topk sampling to introduce some randomness in choosing the most likely next word, given an output set of word scores!

In [ ]:
import torch.nn.functional as F

def generate(rnn, prime_id, int_to_vocab, token_dict, pad_value, predict_len=100, temperature=1.0):
    """
    Generate text using the neural network.

    :param rnn: The PyTorch Module that holds the trained neural network
    :param prime_id: The word id to start the first prediction
    :param int_to_vocab: Dict of word id keys to word values
    :param token_dict: Dict of punctuation tokens keys to punctuation values
    :param pad_value: The value used to pad a sequence
    :param predict_len: The length of text to generate
    :param temperature: Sampling temperature (>0). Lower is more conservative.
    :return: The generated text
    """
    rnn.eval()

    # We do not have a real history at the first step, so we pad the context
    # and place the prime token at the sequence end.
    current_seq = np.full((1, sequence_length), pad_value)
    current_seq[0, -1] = prime_id
    predicted = [int_to_vocab[prime_id]]

    for _ in range(predict_len):
        current_seq_t = torch.as_tensor(current_seq, dtype=torch.long, device=device)
        hidden = rnn.init_hidden(current_seq_t.size(0))
        with torch.no_grad():
            output, _ = rnn(current_seq_t, hidden)  # (1, T, V)

        # Use only the logits at the final timestep for next-token prediction.
        # Temperature controls the sharpness of the distribution.
        logits = output[:, -1, :] / max(temperature, 1e-6)  # (1, V)
        p = F.softmax(logits, dim=-1).detach().cpu()        # (1, V)

        top_k = 5
        p, top_i = p.topk(top_k, dim=-1)                    # both (1, top_k)
        top_i = top_i.squeeze(0).numpy()
        p = p.squeeze(0).numpy()
        word_i = np.random.choice(top_i, p=p / p.sum())

        word = int_to_vocab[word_i]
        predicted.append(word)

        current_seq = np.roll(current_seq, -1, 1)
        current_seq[0, -1] = word_i

    gen_sentences = ' '.join(predicted)

    # Replace punctuation tokens
    for key, token in token_dict.items():
        gen_sentences = gen_sentences.replace(' ' + token.lower(), key)
    gen_sentences = gen_sentences.replace('\n ', '\n')
    gen_sentences = gen_sentences.replace('( ', '(')

    return gen_sentences

### Generate a New Script
Set `gen_length` and choose a `prime_word` (for example: "jerry", "elaine", "george", "kramer").

You can also tune `temperature`:
- lower (`~0.7`) -> safer/more repetitive text
- higher (`~1.2`) -> more diverse/risky text

In [ ]:
# run the cell multiple times to get different results!
gen_length = 400  # modify the length to your preference
prime_word = 'jerry'  # name for starting the script
temperature = 1.0  # try values like 0.7, 1.0, 1.2

pad_word = SPECIAL_WORDS['PADDING']
generated_script = generate(
    trained_rnn,
    vocab_to_int[prime_word + ':'],
    int_to_vocab,
    token_dict,
    vocab_to_int[pad_word],
    gen_length,
    temperature=temperature,
)
print(generated_script)